# XEdu-python 全功能 Jupyter 样例

面向希望亲手验证 XEdu-python 的教师、学生和开发者。本 Notebook 从安装开始，使用真实模型逐项演示视觉、文本、音频和多模态功能。

完成本 Notebook 后，你将能够：

- 安装完整依赖并确认当前 Python 环境；
- 使用 SASU Image2 测试素材；
- 逐格运行每一种内置任务；
- 查看检测框、关键点、OCR、分割、深度、分类和相似度结果；
- 理解哪些功能还需要自己的 checkpoint 或 API Key。

> 首次运行真实模型会自动下载多个 ONNX 文件。建议先单独运行一个功能，确认环境无误后再执行全部单元格。

## 内容导航

1. 定位项目与安装依赖
2. 查看或重新生成 SASU 测试图片
3. 目标检测
4. 姿态与关键点
5. 分类、OCR、生成、分割、深度与驾驶感知
6. 图像、文本、音频与多模态
7. NLP 问答
8. 需要外部模型或 API Key 的功能

## 1. 定位项目与安装依赖

In [ ]:
from pathlib import Path
import subprocess
import sys


def find_project_root(start: Path) -> Path:
    for folder in (start.resolve(), *start.resolve().parents):
        if (folder / "pyproject.toml").exists() and (folder / "XEdu").is_dir():
            return folder
    raise FileNotFoundError("找不到 XEdu-python 项目根目录")


PROJECT_ROOT = find_project_root(Path.cwd())
print("项目目录：", PROJECT_ROOT)
print("Python：", sys.executable)
print("Python 版本：", sys.version.split()[0])

首次运行时，把 `INSTALL_PROJECT` 改为 `True`。安装完成后重启内核，再把它改回 `False`。

等价的终端命令是：

```bash
python3 -m pip install -e ".[all,dev]"
```

In [ ]:
INSTALL_PROJECT = False

if INSTALL_PROJECT:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        f"{PROJECT_ROOT}[all,dev]",
    ])
    print("安装完成，请重启 Notebook 内核。")
else:
    print("未执行安装；如果导入失败，请将 INSTALL_PROJECT 改为 True。")

In [ ]:
import math
import time
import wave

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display
from PIL import Image

from XEdu.hub import Workflow as wf
from XEdu.utils import get_similarity

# False：只检查 Notebook 准备流程；True：真正下载并运行模型。
RUN_MODELS = False

IMAGE_DIR = PROJECT_ROOT / "tests" / "fixtures" / "images"
OUTPUT_DIR = PROJECT_ROOT / "XEdu" / "examples" / "full_feature_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VISION_IMAGE = IMAGE_DIR / "xedu-vision-scene.png"
OCR_IMAGE = IMAGE_DIR / "xedu-ocr-poster.png"
BLANK_IMAGE = IMAGE_DIR / "xedu-blank.png"
ROAD_IMAGE = IMAGE_DIR / "xedu-road-scene.png"

for path in (VISION_IMAGE, OCR_IMAGE, BLANK_IMAGE, ROAD_IMAGE):
    assert path.exists(), f"缺少测试图片：{path}"

print("RUN_MODELS =", RUN_MODELS)
print("支持的任务：", wf.support_task())

## 2. SASU Image2 测试素材

下面四张图片已经通过 `sasu-image2` 的 `gpt-image-2` 生成。若要重新生成，把 `REGENERATE_SASU_IMAGES` 改为 `True`。

- 人物课堂图：检测、姿态、分类、风格迁移、分割、深度、embedding；
- OCR 海报：文字识别；
- 纯白图：空输入和边界情况；
- 道路图：驾驶感知。

In [ ]:
REGENERATE_SASU_IMAGES = False
SASU_DRAW = Path("/Users/apple/.codex/skills/sasu-image2/scripts/draw.py")

image_prompts = {
    VISION_IMAGE: "A realistic educational computer vision test photograph in a classroom lab. One full-body adult teacher faces the camera, face and both hands clearly visible. Include a red ball, blue chair, yellow backpack, green box, laptop and plant. Sharp, clean, no text, no watermark.",
    OCR_IMAGE: "A flat OCR benchmark poster on white. Exact lines: XEDU AI LAB; COMPUTER VISION; DETECTION 123; POSE TEST; TEXT 2026. Large sharp sans-serif text, no extra words, no watermark.",
    BLANK_IMAGE: "A perfectly blank uniform pure white square image, no objects, no text, no shadow, no texture, no watermark.",
    ROAD_IMAGE: "A realistic forward-facing road scene from a car dashboard. Clear lane markings, drivable road, one car ahead, pedestrian on sidewalk, traffic sign, daylight, no overlay, no watermark.",
}

if REGENERATE_SASU_IMAGES:
    for output_path, prompt in image_prompts.items():
        size = "1024x1024" if output_path == BLANK_IMAGE else "1536x1024"
        subprocess.check_call([
            sys.executable,
            str(SASU_DRAW),
            "--prompt", prompt,
            "--size", size,
            "--quality", "high",
            "--out", str(output_path),
            "--force",
        ])
else:
    print("使用已生成的 SASU 测试图。")

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(14, 9))
assets = [
    (VISION_IMAGE, "人物与物体"),
    (OCR_IMAGE, "OCR 海报"),
    (BLANK_IMAGE, "纯白边界图"),
    (ROAD_IMAGE, "道路场景"),
]
for axis, (path, title) in zip(axes.flat, assets):
    axis.imshow(Image.open(path))
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()

In [ ]:
# 为图像着色任务准备灰度输入。
gray = cv2.imread(str(VISION_IMAGE), cv2.IMREAD_GRAYSCALE)
GRAY_IMAGE = OUTPUT_DIR / "xedu-vision-gray.png"
cv2.imwrite(str(GRAY_IMAGE), gray)

# 为音频任务生成两个一秒钟的 WAV 音调。
def make_tone(path: Path, frequency: float, sample_rate: int = 48000) -> str:
    samples = np.arange(sample_rate, dtype=np.float32) / sample_rate
    waveform = 0.25 * np.sin(2 * math.pi * frequency * samples)
    pcm = (waveform * 32767).astype(np.int16)
    with wave.open(str(path), "wb") as handle:
        handle.setnchannels(1)
        handle.setsampwidth(2)
        handle.setframerate(sample_rate)
        handle.writeframes(pcm.tobytes())
    return str(path)


AUDIO_440 = make_tone(OUTPUT_DIR / "tone-440.wav", 440)
AUDIO_880 = make_tone(OUTPUT_DIR / "tone-880.wav", 880)
display(Audio(AUDIO_440))

In [ ]:
def show_bgr(image, title: str, figsize=(10, 6)):
    # 仅负责在 Notebook 中显示 OpenCV BGR 图片。
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()


def waiting(name: str):
    print(f"{name} 尚未执行：把 RUN_MODELS 改为 True 后重新运行本单元格。")

## 3. 目标检测

### 3.1 人体检测 `det_body`

In [ ]:
if RUN_MODELS:
    det_body = wf(task="det_body")
    body_boxes, body_image = det_body.inference(
        data=str(VISION_IMAGE), img_type="cv2", thr=0.20
    )
    print("人体框：", body_boxes)
    print(det_body.format_output(lang="zh", isprint=False))
    show_bgr(body_image, "人体检测 det_body")
else:
    waiting("det_body")

### 3.2 大模型人体检测 `det_body_l`

In [ ]:
if RUN_MODELS:
    det_body_l = wf(task="det_body_l")
    body_l_boxes, body_l_image = det_body_l.inference(
        data=str(VISION_IMAGE), img_type="cv2", thr=0.20
    )
    print("大模型人体框：", body_l_boxes)
    show_bgr(body_l_image, "大模型人体检测 det_body_l")
else:
    waiting("det_body_l")

### 3.3 COCO 目标检测 `det_coco`

In [ ]:
if RUN_MODELS:
    det_coco = wf(task="det_coco")
    coco_boxes, coco_image = det_coco.inference(
        data=str(VISION_IMAGE), img_type="cv2", thr=0.20
    )
    print(det_coco.format_output(lang="zh", isprint=False))
    show_bgr(coco_image, "COCO 目标检测 det_coco")
else:
    waiting("det_coco")

### 3.4 大模型 COCO 检测 `det_coco_l`

In [ ]:
if RUN_MODELS:
    det_coco_l = wf(task="det_coco_l")
    coco_l_boxes, coco_l_image = det_coco_l.inference(
        data=str(VISION_IMAGE), img_type="cv2", thr=0.20
    )
    print(det_coco_l.format_output(lang="zh", isprint=False))
    show_bgr(coco_l_image, "大模型 COCO 检测 det_coco_l")
else:
    waiting("det_coco_l")

### 3.5 手部检测 `det_hand`

In [ ]:
if RUN_MODELS:
    det_hand = wf(task="det_hand")
    hand_boxes, hand_image = det_hand.inference(
        data=str(VISION_IMAGE), img_type="cv2", thr=0.15
    )
    print("手部框：", hand_boxes)
    show_bgr(hand_image, "手部检测 det_hand")
else:
    waiting("det_hand")

### 3.6 人脸检测 `det_face`

In [ ]:
if RUN_MODELS:
    det_face = wf(task="det_face")
    face_boxes, face_image = det_face.inference(
        data=str(VISION_IMAGE), img_type="cv2", thr=0.30
    )
    print(det_face.format_output(lang="zh", isprint=False))
    show_bgr(face_image, "人脸检测 det_face")
else:
    waiting("det_face")

## 4. 姿态与关键点

In [ ]:
# 这些坐标针对 1536×1024 的 SASU 人物图。
PERSON_BBOX = np.array([560, 35, 960, 975], dtype=np.float32)
LEFT_HAND_BBOX = np.array([555, 165, 690, 350], dtype=np.float32)
FACE_BBOX = np.array([700, 55, 835, 205], dtype=np.float32)

### 4.1 人体 17 点 `pose_body17`

In [ ]:
if RUN_MODELS:
    pose17 = wf(task="pose_body17")
    keypoints17, pose17_image = pose17.inference(
        data=str(VISION_IMAGE), img_type="cv2", bbox=PERSON_BBOX
    )
    print("关键点形状：", np.asarray(keypoints17).shape)
    show_bgr(pose17_image, "人体 17 点")
else:
    waiting("pose_body17")

### 4.2 大模型人体 17 点 `pose_body17_l`

In [ ]:
if RUN_MODELS:
    pose17_l = wf(task="pose_body17_l")
    keypoints17_l, pose17_l_image = pose17_l.inference(
        data=str(VISION_IMAGE), img_type="cv2", bbox=PERSON_BBOX
    )
    print("关键点形状：", np.asarray(keypoints17_l).shape)
    show_bgr(pose17_l_image, "大模型人体 17 点")
else:
    waiting("pose_body17_l")

### 4.3 人体 26 点 `pose_body26`

In [ ]:
if RUN_MODELS:
    pose26 = wf(task="pose_body26")
    keypoints26, pose26_image = pose26.inference(
        data=str(VISION_IMAGE), img_type="cv2", bbox=PERSON_BBOX
    )
    print("关键点形状：", np.asarray(keypoints26).shape)
    show_bgr(pose26_image, "人体 26 点")
else:
    waiting("pose_body26")

### 4.4 手部 21 点 `pose_hand21`

In [ ]:
if RUN_MODELS:
    hand21 = wf(task="pose_hand21")
    hand_keypoints, hand_pose_image = hand21.inference(
        data=str(VISION_IMAGE), img_type="cv2", bbox=LEFT_HAND_BBOX
    )
    print("手部关键点形状：", np.asarray(hand_keypoints).shape)
    show_bgr(hand_pose_image, "手部 21 点")
else:
    waiting("pose_hand21")

### 4.5 全身 133 点 `pose_wholebody133`

In [ ]:
if RUN_MODELS:
    wholebody = wf(task="pose_wholebody133")
    whole_keypoints, whole_image = wholebody.inference(
        data=str(VISION_IMAGE), img_type="cv2", bbox=PERSON_BBOX
    )
    print("全身关键点形状：", np.asarray(whole_keypoints).shape)
    show_bgr(whole_image, "全身 133 点")
else:
    waiting("pose_wholebody133")

### 4.6 人脸关键点 MobileNet-106

In [ ]:
if RUN_MODELS:
    face_landmark = wf(task="pose_face_landmark")
    face_points, face_landmark_image = face_landmark.inference(
        data=str(VISION_IMAGE), img_type="cv2", bbox=FACE_BBOX
    )
    print(face_landmark.format_output(lang="zh", isprint=False))
    show_bgr(face_landmark_image, "人脸关键点 MobileNet-106")
else:
    waiting("pose_face_landmark")

### 4.7 人脸关键点 PIPNet/WFLW-98

In [ ]:
if RUN_MODELS:
    face_pipnet = wf(
        task="pose_face_landmark",
        model_id="pose_face_landmark-pipnet98-wflw",
    )
    face98_points, face98_image = face_pipnet.inference(
        data=str(VISION_IMAGE), img_type="cv2", bbox=FACE_BBOX
    )
    print(face_pipnet.format_output(lang="zh", isprint=False))
    show_bgr(face98_image, "人脸关键点 PIPNet/WFLW-98")
else:
    waiting("pose_face_landmark PIPNet-98")

`pose_face106` 是历史兼容入口，自动下载已禁用，因为旧地址曾指向错误模型。必须提供自己的本地 checkpoint：

```python
pose_face106 = wf(task="pose_face106", checkpoint="/path/to/face106.onnx")
points, image = pose_face106.inference(str(VISION_IMAGE), img_type="cv2", bbox=FACE_BBOX)
```

## 5. 分类、OCR、生成、分割、深度与驾驶感知

### 5.1 ImageNet 分类 `cls_imagenet`

In [ ]:
if RUN_MODELS:
    classifier = wf(task="cls_imagenet")
    class_scores, class_image = classifier.inference(
        data=str(VISION_IMAGE), img_type="cv2"
    )
    print(classifier.format_output(lang="zh", isprint=False))
    show_bgr(class_image, "ImageNet 分类输入")
else:
    waiting("cls_imagenet")

### 5.2 OCR `ocr`

In [ ]:
if RUN_MODELS:
    ocr = wf(task="ocr")
    ocr_result, ocr_image = ocr.inference(
        data=str(OCR_IMAGE), img_type="cv2"
    )
    print("OCR 原始结果：", ocr_result)
    print("OCR 格式化结果：", ocr.format_output(lang="zh", isprint=False))
    show_bgr(ocr_image, "OCR 识别结果")
else:
    waiting("ocr")

### 5.3 五种风格迁移 `gen_style`

In [ ]:
if RUN_MODELS:
    styles = ["mosaic", "candy", "rain-princess", "udnie", "pointilism"]
    figure, axes = plt.subplots(1, len(styles), figsize=(20, 5))
    for axis, style_name in zip(axes, styles):
        style_model = wf(task="gen_style", style=style_name)
        styled, _ = style_model.inference(
            data=str(VISION_IMAGE), img_type="cv2"
        )
        axis.imshow(cv2.cvtColor(styled, cv2.COLOR_BGR2RGB))
        axis.set_title(style_name)
        axis.axis("off")
    plt.tight_layout()
else:
    waiting("gen_style")

### 5.4 图像着色 `gen_color`

In [ ]:
if RUN_MODELS:
    colorizer = wf(task="gen_color")
    colored, _ = colorizer.inference(data=str(GRAY_IMAGE), img_type="cv2")
    figure, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(Image.open(GRAY_IMAGE), cmap="gray")
    axes[0].set_title("灰度输入")
    axes[1].imshow(cv2.cvtColor(colored, cv2.COLOR_BGR2RGB))
    axes[1].set_title("着色结果")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
else:
    waiting("gen_color")

### 5.5 Segment Anything `segment_anything`

In [ ]:
if RUN_MODELS:
    segmenter = wf(task="segment_anything")
    masks, segmented_image = segmenter.inference(
        data=str(VISION_IMAGE),
        mode="box",
        prompt=PERSON_BBOX,
        img_type="cv2",
    )
    print("掩码形状：", np.asarray(masks).shape)
    show_bgr(segmented_image, "SAM 人物分割")
else:
    waiting("segment_anything")

### 5.6 深度估计 `depth_anything`

In [ ]:
if RUN_MODELS:
    depth_model = wf(task="depth_anything")
    depth_map, _ = depth_model.inference(
        data=str(VISION_IMAGE), img_type="cv2"
    )
    plt.figure(figsize=(10, 6))
    plt.imshow(np.squeeze(depth_map), cmap="inferno")
    plt.colorbar(label="relative depth")
    plt.title("Depth Anything")
    plt.axis("off")
    plt.show()
else:
    waiting("depth_anything")

### 5.7 驾驶感知 `drive_perception`

In [ ]:
if RUN_MODELS:
    driving = wf(task="drive_perception")
    driving_result, driving_image = driving.inference(
        data=str(ROAD_IMAGE), img_type="cv2", thr=0.20
    )
    boxes, lane_mask, area_mask = driving_result
    print("车辆/交通目标框：", np.asarray(boxes).shape)
    print("车道线掩码：", np.asarray(lane_mask).shape)
    print("可行驶区域掩码：", np.asarray(area_mask).shape)
    show_bgr(driving_image, "驾驶感知结果")
else:
    waiting("drive_perception")

## 6. 图像、文本、音频与多模态

### 6.1 图像 embedding `embedding_image`

In [ ]:
if RUN_MODELS:
    image_embedder = wf(task="embedding_image")
    image_embeddings = image_embedder.inference([
        str(VISION_IMAGE), str(OCR_IMAGE), str(BLANK_IMAGE), str(ROAD_IMAGE)
    ])
    print("图像 embedding 形状：", image_embeddings.shape)
    image_similarity = get_similarity(
        image_embeddings, image_embeddings, method="cosine", use_softmax=False
    )
    print("图像两两相似度：")
    print(np.round(image_similarity, 3))
else:
    waiting("embedding_image")

### 6.2 文本 embedding `embedding_text`

In [ ]:
if RUN_MODELS:
    text_embedder = wf(task="embedding_text")
    texts = ["a teacher in a classroom", "a road with cars", "a white blank image"]
    text_embeddings = text_embedder.inference(texts)
    print("文本 embedding 形状：", text_embeddings.shape)
else:
    waiting("embedding_text")

### 6.3 原型文本分类 `cls_text`

In [ ]:
if RUN_MODELS:
    text_classifier = wf(task="cls_text")
    text_class_result = text_classifier.inference(
        data="image recognition lesson",
        prototypes={
            "人工智能": ["computer vision", "machine learning"],
            "体育": ["running", "basketball"],
        },
    )
    print(text_class_result)
else:
    waiting("cls_text")

### 6.4 音频 embedding `embedding_audio`

In [ ]:
if RUN_MODELS:
    audio_embedder = wf(task="embedding_audio")
    audio_embeddings = audio_embedder.inference([AUDIO_440, AUDIO_880])
    print("音频 embedding 形状：", audio_embeddings.shape)
else:
    waiting("embedding_audio")

### 6.5 原型音频分类 `cls_audio`

In [ ]:
if RUN_MODELS:
    audio_classifier = wf(task="cls_audio")
    audio_class_result = audio_classifier.inference(
        data=AUDIO_440,
        prototypes={"440Hz": AUDIO_440, "880Hz": AUDIO_880},
    )
    print(audio_class_result)
else:
    waiting("cls_audio")

### 6.6 音频关键词/事件检测 `det_audio_keyword`

In [ ]:
if RUN_MODELS:
    keyword_detector = wf(task="det_audio_keyword")
    keyword_result = keyword_detector.inference(
        data=AUDIO_440,
        prototypes={"440Hz": AUDIO_440, "880Hz": AUDIO_880},
        threshold=0.0,
        top_k=2,
    )
    print(keyword_result)
else:
    waiting("det_audio_keyword")

### 6.7 图文匹配 `match_image_text`

In [ ]:
if RUN_MODELS:
    matcher = wf(task="match_image_text")
    match_result = matcher.inference(
        data=[str(VISION_IMAGE), str(ROAD_IMAGE), str(BLANK_IMAGE)],
        texts=["a teacher in a classroom", "a road with cars", "a blank white image"],
    )
    print("最佳匹配：", match_result["matches"])
    print("相似度矩阵：")
    print(np.round(match_result["similarities"], 3))
else:
    waiting("match_image_text")

## 7. NLP 问答 `nlp_qa`

In [ ]:
if RUN_MODELS:
    qa = wf(task="nlp_qa")
    context = (
        "XEdu-python is an AI education toolkit for K-12 classrooms. "
        "It provides computer vision, audio, text, and multimodal inference APIs."
    )
    qa_result = qa.inference(
        data="What does XEdu-python provide?",
        context=context,
    )
    print("问答结果：", qa_result)
    print(qa.format_output(lang="en", isprint=False, show_context=True))
else:
    waiting("nlp_qa")

## 8. 需要外部模型或 API Key 的功能

以下接口无法仅依靠安装包自动完成真实推理，应替换为你自己的文件或密钥。

In [ ]:
# MMEdu 导出的 ONNX
# mmedu = wf(task="mmedu", checkpoint="/path/to/exported.onnx")
# result = mmedu.inference(str(VISION_IMAGE))

# BaseNN 导出的 ONNX
# basenn = wf(task="basenn", checkpoint="/path/to/basenn.onnx")
# result = basenn.inference(np.array([[1.0, 2.0]], dtype=np.float32))

# BaseML 导出的 PKL
# baseml = wf(task="baseml", checkpoint="/path/to/model.pkl")
# result = baseml.inference([[1.0, 2.0]])

# 自定义 ONNX：还需要自己编写 preprocess 和 postprocess
# custom = wf(task="custom", checkpoint="/path/to/custom.onnx")
# result = custom.inference(str(VISION_IMAGE), preprocess=my_preprocess, postprocess=my_postprocess)

# 模型仓库
# repo_model = wf(repo="owner/repository", download_path="repo_cache")

# LLM：不要把 API Key 写进 Notebook，使用环境变量
# import os
# from XEdu.LLM import Client
# client = Client(provider="qwen", api_key=os.environ["QWEN_API_KEY"])
# response = client.inference("请介绍 XEdu-python", stream=False)

## 常见问题

- `No module named pytest/soundfile/rapidocr_onnxruntime`：重新安装 `.[all,dev]`。
- 模型下载慢：模型会缓存在 `~/.cache/XEdu/hub`，首次运行后无需重复下载。
- 检测结果为空不等于程序失败：阈值、测试图和模型类别都会影响检测数量。
- `pose_face106` 报错：这是预期行为，必须提供正确的本地 checkpoint。
- pytest 中的 `slow` 仅表示 wheel 打包测试耗时较长，与本 Notebook 的真实推理样例无关。

## 练习

任选一张自己的课堂照片，替换 `VISION_IMAGE`，依次运行 `det_body`、`pose_body17` 和 `segment_anything`，比较三种任务的输出差异。

In [ ]:
# 练习代码模板
MY_IMAGE = VISION_IMAGE  # 替换成 Path("/path/to/your-image.jpg")

# detector = wf(task="det_body")
# boxes, detected = detector.inference(str(MY_IMAGE), img_type="cv2")
# pose = wf(task="pose_body17")
# keypoints, posed = pose.inference(str(MY_IMAGE), img_type="cv2", bbox=boxes[0])
# show_bgr(posed, "我的课堂照片姿态结果")